In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

df = pd.read_csv("../data/raw/fraud_oracle.csv")

X = df.drop(columns=[
    "FraudFound_P",
    "PolicyNumber",
    "RepNumber"
])

y = df["FraudFound_P"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.copy()
X_test = X_test.copy()

X_train["Age"] = X_train["Age"].replace(0, np.nan)
X_test["Age"] = X_test["Age"].replace(0, np.nan)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (12336, 30)
Testing: (3084, 30)


In [2]:
categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

categorical_features += ["Year", "DriverRating"]

numerical_features = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in categorical_features
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("MLP preprocessing ready")

MLP preprocessing ready


In [3]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("Non-fraud weight:", sample_weights[y_train == 0][0])
print("Fraud weight:", sample_weights[y_train == 1][0])

Non-fraud weight: 0.5318158303155717
Fraud weight: 8.357723577235772


In [4]:
mlp_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        max_iter=300,
        random_state=42
    ))
])

mlp_model.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

mlp_train_pred = mlp_model.predict(X_train)

print("MLP — Training Results")
print(classification_report(
    y_train,
    mlp_train_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, mlp_train_pred))

MLP — Training Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     11598
           1       1.00      1.00      1.00       738

    accuracy                           1.00     12336
   macro avg       1.00      1.00      1.00     12336
weighted avg       1.00      1.00      1.00     12336


Confusion Matrix:
[[11598     0]
 [    0   738]]


In [10]:
regularized_mlp = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.01,
        max_iter=200,
        random_state=42
    ))
])

print("Regularized MLP created successfully")

Regularized MLP created successfully


In [12]:

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)
import pandas as pd

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

regularized_cv_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    fold_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_fold_train
    )

    fold_model = clone(regularized_mlp)

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        classifier__sample_weight=fold_weights
    )

    predictions = fold_model.predict(X_fold_val)
    probabilities = fold_model.predict_proba(X_fold_val)[:, 1]

    regularized_cv_results.append({
        "Fold": fold,
        "Precision": precision_score(
            y_fold_val,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_fold_val,
            predictions,
            zero_division=0
        ),
        "Average Precision": average_precision_score(
            y_fold_val,
            probabilities
        )
    })

regularized_cv_df = pd.DataFrame(regularized_cv_results)

print("Regularized MLP - Cross-Validation Results")
display(regularized_cv_df.round(4))

print("\nRegularized MLP - Mean Cross-Validation Results")
print(
    regularized_cv_df
    .drop(columns="Fold")
    .mean()
    .round(4)
)


Regularized MLP - Cross-Validation Results


,Fold,Precision,Recall,F1,Average Precision
0,1,0.1935,0.2027,0.1980,0.1426
1,2,0.2152,0.2313,0.2230,0.1793
2,3,0.2121,0.2857,0.2435,0.1918
3,4,0.2011,0.2432,0.2202,0.1542
4,5,0.1561,0.1824,0.1682,0.1500



Regularized MLP - Mean Cross-Validation Results
Precision            0.1956
Recall               0.2291
F1                   0.2106
Average Precision    0.1636
dtype: float64
